# Phase 2 matched experiment: implementation

## Purpose and population

This notebook runs the matched, non-clustered Phase 2 geometric monitoring experiment. It is not the historical full-dataset workflow or the FMIC/SMIC clustering experiment. The shared `scripts/shared_monitoring_engine.py` owns preparation, monitoring decisions, the offline correctness oracle, and publication; this notebook orchestrates the run and checks its saved exports.

The experiment uses the validated Phase 1 aligned temperature file at `artifacts/phase1_operational/phase1_aligned_temperature.csv`, the canonical `config/project_config.yaml`, and the fixed A/C/D/E/F/G cohort over 19,984 consecutive UTC minute buckets from 2022-02-24 18:50 through 2022-03-10 15:53 inclusive. Requiring all six finite readings at every bucket preserves the same population and denominator for a fair matched comparison; missing values are rejected, not imputed. The all-device Phase 1 average is not necessarily the same series as this six-sensor mean.

Shared preparation derives the six-sensor mean's empirical P90 from this population. The current saved run is approximately 22.42866665 °C, but this value is evidence, not a hard-coded decision constant or a separately certified physical safety limit. The configuration names the threshold policy and source series rather than storing an authoritative numeric P90.

## Outputs and execution

Run every cell from a clean kernel. The `publish` call writes the matched summary to `artifacts/phase2_outputs/phase2_summary.json` and completed-synchronization events to `artifacts/phase2_operational/phase2_events.csv`; these are production destinations, not scratch copies. The separate `artifacts/phase2_operational/phase2_validation.json` receipt is owned by the independent validation workflow, so successful publication does not prove that the receipt is current.

The stages resolve the repository root, validate configuration and preparation evidence, execute and gate the shared monitor, then verify the saved exports. Missing or stale inputs, failed correctness/accounting checks, and export mismatches stop execution. Run `phase2_test.ipynb` after publication: it runs the regression suite and isolated reproduction before issuing the independent receipt. This fixed-cohort research run does not model a live physical network or the original full-dataset comparison.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd

# Resolve the project from the starting directory; reject unrelated folders
# instead of guessing at configuration or artifact paths.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "config" / "project_config.yaml").is_file():
    parent = REPO_ROOT.parent
    if parent == REPO_ROOT:
        raise FileNotFoundError("Run this notebook from the sensor_project repository.")
    REPO_ROOT = parent

# Enable imports from this checkout; this is project path setup, not a general
# security boundary for arbitrary Python modules.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The shared engine owns configuration loading, preparation, and publication;
# POLICY_ID names its two-sided policy and is not an algorithm in this notebook.
from scripts.shared_monitoring_engine import POLICY_ID, load_phase3a_config, publish

CONFIG_PATH = REPO_ROOT / "config" / "project_config.yaml"
INPUT_DIR = REPO_ROOT / "artifacts"
# These are the canonical production destinations used by publish().
SUMMARY_PATH = INPUT_DIR / "phase2_outputs" / "phase2_summary.json"
EVENTS_PATH = INPUT_DIR / "phase2_operational" / "phase2_events.csv"

def sha256(path):
    """Hash the exact bytes of an existing file given by ``path`` in 1 MiB blocks.

    Return the lowercase hexadecimal SHA-256 digest used to identify the saved
    event export. A missing or unreadable path raises the underlying file error;
    the digest checks byte identity, not the scientific meaning of the file.
    """
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print(f"Repository: {REPO_ROOT}")
print(f"Shared monitoring policy: {POLICY_ID}")

## 1. Validate inputs and resolve the approved policy

The canonical configuration identifies the approved cohort, inclusive UTC endpoints, 60-second cadence, aligned input, source series, and derived-P90 policy. It does not store a numeric P90 as authoritative configuration. `prepare_monitoring(REPO_ROOT)` validates the Phase 1 producer and source hashes, selects all 19,984 buckets, computes each complete six-sensor mean, and derives the full-precision P90; the result retains the population and provenance evidence without modifying configuration or artifacts.

Each selected reading must be finite, each epoch must match its UTC timestamp, and timestamps must be unique, consecutive, and include both endpoints. A stale producer or invalid population stops the run instead of substituting values. The displayed metadata is a review aid only. The aggregate all-device Phase 1 series is not interchangeable with the matched six-sensor series.

In [ ]:
# Shared preparation validates source freshness and recalculates the cohort P90;
# it does not persist the displayed metadata or change the configuration.
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing experiment configuration: {CONFIG_PATH}")
config = load_phase3a_config(REPO_ROOT)
threshold_config = config["threshold"]
input_path = REPO_ROOT / config["input_source"]
from scripts.shared_monitoring_engine import prepare_monitoring
monitoring_preparation = prepare_monitoring(REPO_ROOT)
threshold = monitoring_preparation["p90_value"]
if config["experiment_name"] != "phase3a_stable_cohort":
    raise ValueError("Only the approved phase3a_stable_cohort experiment may run here.")

# This preflight display reports what is configured; it is not validation or
# independent test evidence.
print(json.dumps({
    "experiment": config["experiment_name"],
    "sensors": config["selected_sensor_ids"],
    "start_utc_inclusive": config["start_timestamp_utc"],
    "end_utc_inclusive": config["end_timestamp_utc"],
    "expected_buckets": config["expected_bucket_count"],
    "threshold_policy": POLICY_ID,
    "threshold_series": threshold_config["source_series"],
    "threshold_p90": threshold,
}, indent=2))

## 2. Run the shared monitoring engine

Each sensor compares its current temperature with its own value from the last completed synchronization, not with the prior-minute value or the old global mean. Its signed deviation is current reading minus that synchronized reference. The local allowance is the absolute distance between the last synchronized global mean and the cohort P90; the two-sided trigger is `abs(deviation) >= allowance`. A negative deviation can therefore trigger, equality triggers, and a zero allowance can trigger even when the reading is unchanged.

One or more local triggers can cause one root synchronization for a bucket. Initialization and membership changes can also force a synchronization; those causes are not themselves global threshold violations. A completed synchronization updates the root-known label and every observed sensor's reference and allowance for the next bucket. SAFE means the synchronized mean is strictly below P90; VIOLATED includes equality or above. The root-known label is the last verified state, not a continuously delivered mean. The observed mean at each bucket is used by an offline oracle to assess sampled-bucket correctness, not supplied to the root without messages. This fixed experiment has no departures or re-entries, although the reusable engine supports availability changes.

The engine counts logical transmissions between the separate root and sensors: trigger notifications, root requests, sensor responses, and state updates are distinct categories. A synchronization event is not the same as a trigger or threshold violation, and no self-message is counted. Category totals reconcile to `total_messages`; cause-attribution counters can overlap and are not additive categories. The notebook is an orchestration and export-integrity check, not a prediction model or a second state machine.

In [ ]:
# Production side effect: publish() executes the experiment, writes the event
# CSV first, then the summary that references its exact digest. This is not a
# read-only calculation and does not create an independent validation receipt.
summary = publish(REPO_ROOT)
# correctness_status and false-state counts compare the last root-known label
# with the offline oracle at each sampled bucket; the evaluation count should
# equal the matched population. Message reconciliation checks category totals,
# not payload bytes, energy, latency, or between-bucket/network resilience.
if summary["correctness_status"] != "passed":
    raise RuntimeError(f"Independent root-state correctness gate failed: {summary}")
if summary["message_counts_reconcile"] is not True:
    raise RuntimeError("Phase 2 message categories do not reconcile to total transmissions.")
if summary["root_state_evaluation_count"] != config["expected_bucket_count"]:
    raise RuntimeError("Not every matched bucket received an independent root-state check.")
if summary["false_safe_count"] or summary["false_violated_count"] or summary["root_state_not_evaluable_count"]:
    raise RuntimeError("The matched root-state correctness gate reported errors.")
if summary["total_messages"] != sum(summary["message_category_totals"].values()):
    raise RuntimeError("Phase 2 category totals do not equal total transmissions.")

# The table reports bucket/count measures; it is not a trained classifier or
# prediction result. Temperatures in the source are degrees Celsius.
display(pd.DataFrame([
    {"measure": "Matched buckets", "value": summary["bucket_count"]},
    {"measure": "Threshold violations", "value": summary["global_threshold_violation_count"]},
    {"measure": "Root synchronizations", "value": summary["resynchronization_count"]},
    {"measure": "Total transmissions", "value": summary["total_messages"]},
    {"measure": "False-safe root states", "value": summary["false_safe_count"]},
    {"measure": "False-violated root states", "value": summary["false_violated_count"]},
    *[
        {"measure": name.replace("_message_count", " transmissions"), "value": count}
        for name, count in summary["message_category_totals"].items()
    ],
]))

## 3. Verify current exports

The summary and completed-synchronization event table are the current Phase 2 outputs; the event CSV has one row per completed root synchronization, not one row per minute. The summary binds the exact event-file bytes by SHA-256, while the independent receipt is written later by `phase2_test.ipynb`.

These checks require both canonical files, compare the saved JSON with the in-memory publication result, compare the event bytes with the summary's digest, reconcile event rows and transmission totals, and restrict completed-event root labels to SAFE or VIOLATED. They detect missing, stale, partial, or mismatched exports; hashing does not establish scientific correctness. The compact production files do not contain the full per-minute, full-precision trace. Independent regression tests and isolated reproduction remain necessary.

In [ ]:
# Confirm the saved publication matches this run. The event digest identifies
# exact file bytes; it does not replace the test notebook's independent checks.
if not SUMMARY_PATH.is_file() or not EVENTS_PATH.is_file():
    raise FileNotFoundError("The matched Phase 2 summary or event export is missing.")
saved_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
events = pd.read_csv(EVENTS_PATH, keep_default_na=False)
if saved_summary != summary:
    raise ValueError("The saved matched summary differs from the completed engine run.")
if sha256(EVENTS_PATH) != summary["events_sha256"]:
    raise ValueError("The saved matched event file does not match the summary hash.")
if len(events) != summary["resynchronization_count"]:
    raise ValueError("Event rows do not reconcile to completed synchronizations.")
if int(events["total_message_count"].sum()) != summary["total_messages"]:
    raise ValueError("Event-level transmissions do not reconcile to the summary.")
if set(events["root_known_state"]) - {"SAFE", "VIOLATED"}:
    raise ValueError("A completed synchronization has an invalid root-known state.")

print(f"Summary: {SUMMARY_PATH.relative_to(REPO_ROOT)}")
print(f"Events: {EVENTS_PATH.relative_to(REPO_ROOT)} ({len(events):,} rows)")
print(f"Event SHA-256: {summary['events_sha256']}")
print("Matched Phase 2 implementation and export checks passed.")